<a href="https://colab.research.google.com/github/reejungkim/Pipeline/blob/main/prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Load up packages
import os
import pandas as pd
import numpy as np
import sklearn

#visualization lisb
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set_palette('Set2')

#split data set
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score

#feature selection
from sklearn import feature_selection

#preprocessing libs
#scaler
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
#pipeline
from sklearn.pipeline import Pipeline
from sklearn.pipeline import FeatureUnion
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.base import BaseEstimator, TransformerMixin
#label encoders
from sklearn.preprocessing import LabelEncoder,  OrdinalEncoder, OneHotEncoder

#Models
#classifiers
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
#regressors
from sklearn.linear_model import LinearRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor , GradientBoostingRegressor


#model evaluation libs
#classifier evaluation
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
#regressor evaluation
from sklearn.metrics import r2_score, mean_squared_error

#number of features optimization
from sklearn.model_selection import StratifiedKFold

#Hyper-parameter optimization
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV


# Load data set

In [ ]:
# Load data
#df = pd.read_csv('bankruptcy_Train.csv')

#df = sns.load_dataset('diamonds')
#df.head()


# Seaborn ???? ??
sns.get_dataset_names()

df = sns.load_dataset('titanic')
df.head()

In [ ]:
features = df.loc[:, df.columns!='survived']
output = df.loc[:, df.columns=='survived']

# Split data set

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, output, 
                                                    test_size=0.2,
                                                    random_state=52)

# Exploratory Data Analysis

In [ ]:
pd.concat( [ df.dtypes.rename('dtype') , df.iloc[0].rename('example') ], axis=1 ) 

In [ ]:
labels = [0,1]
sizes = [y_train['survived'].value_counts()[0],
         y_train['survived'].value_counts()[1]]

colors = sns.color_palette('pastel')
plt.pie(sizes, labels=labels,  autopct='%.0f%%', colors=colors)
plt.show()

In [ ]:
# checking for sparse columns 
#(df.isnull().sum()/ len(df).sort_values(ascending=False)[:30]
df_null = pd.DataFrame((X_train.isnull().sum()/ len(X_train)), columns={"%missing"}).sort_values(by="%missing", ascending=False)
df_null.loc[df_null['%missing']>=.5]

should check if the sparse columns refers to zero or n/a before dropping the columns
 

In [ ]:
X_train.drop(['deck'], axis=1, inplace=True)

Drop columns logically shouldn't be used

In [ ]:
X_train.drop(['alive'], axis=1, inplace=True )

In [ ]:
df_no_var = features.std().sort_values(ascending=True)
df_no_var#.loc[df_no_var<=0.2]

In [ ]:
#df.drop(['Attr1', 'Attr2'], axis=1)

In [ ]:
df.info()

In [ ]:
cols_obj = X_train.select_dtypes(include=['object','category', 'boolean']).columns

In [ ]:
for col in cols_obj:
    print('{}: {}'.format(col, df[col].unique()  ))

# Pipeline

In [ ]:
class AgeCustomImputer(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    return self
  def transform(self, X, y=None):
    X['age'] = X.groupby('class')['age'].transform( lambda x: x.fillna(x.median()))
    X['age'] = X['age'].astype('int')
    return X

In [ ]:
class NameTransformer(BaseEstimator, TransformerMixin):
  def __init__(self):
    self.name_ls=[]
  def fit(self, X, y=None):
    return self
  def transform(self, X, y=None):
    for name_idx in range(len(X['name'])):
      self.name_ls.append(X['name'][name_idx].split(',')[0])
    X['name'] = self.name_ls
    return X

make_column_transformer(
    (StandardScaler(), ['carat', 'depth']),
    (OneHotEncoder(), ['cut']),
    remainder='passthrough'
)

In [ ]:
num_pipeline = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
        ]
)


cat_pipeline = Pipeline(
    steps = [ 
        #('imputer_constant', SimpleImputer(strategy='constant', fill_value='none'),
        ("imputer", SimpleImputer(strategy="most_frequent")),
        #("encoder", OneHotEncoder(handle_unknown="ignore")) 
        ("encoder", OrdinalEncoder()) 
        ]
)


In [ ]:
# ??? ??? ???? ????.
num_x_training = X_train.select_dtypes(include=['number', 'int64', 'float64'] )\
                            .columns \
                            .to_list()
# ??? ??? ???? ????.
cat_x_training = X_train.select_dtypes(include=['object', 'category', 'boolean'])\
                            .columns \
                            .to_list()

# ColumnTranformer ? ? ??? ?? ??
preprocessor = ColumnTransformer([
    ('numeric', num_pipeline, num_x_training),
    ('categorical', cat_pipeline, cat_x_training)  #['sex', 'embarked', 'embark_town']) 
], remainder='drop')

# ColumnTranformer ? ? ??? ?? ??
full_pipeline = Pipeline(steps=[
        ("age imputer", AgeCustomImputer()),
        ('preprocessor', preprocessor)
    ]
)

In [ ]:
sklearn.set_config(display="diagram")
full_pipeline.fit(X_train)

sel = feature_selection.VarianceThreshold() #removes all features whose variance doesn?t meet some threshold. By default it removes features with zero variance or features that have the same value for all samples.
train_variance =  sel.fit_transform(preprocessor.transform(X_train))
train_variance

In [ ]:
model_candidates = [   
{
    'label':'Random forest classifier',
    'model': RandomForestClassifier(),
},
{
    'label':'GBM classifier',
    'model':GradientBoostingClassifier(),
},
{
    'label': 'XGBoost Classifier',
    'model': XGBClassifier(eval_metric='mlogloss', use_label_encoder=False),#max_depth=10, n_estimators=300, use_label_encoder=False),
#  # 'model': GradientBoostingClassifier(n_estimators=300,max_depth=10)
},
{
    'label':'LightGBM',
    'model': LGBMClassifier()
},
{
    'label':'Logistics',
    'model': LogisticRegression()
}

]

In [ ]:
full_pipeline.fit(X_train)

X_train_piped = full_pipeline.transform(X_train)
X_test_piped = full_pipeline.transform(X_test) 

full_pipeline.named_steps

In [ ]:
# Below for loop iterates through your models list
for m in model_candidates:
    model = m['model'] # select the model
    model.fit( X_train_piped , y_train) # train the model
    y_pred=model.predict(X_test_piped) # predict the test data
# Compute False postive rate, and True positive rate
    fpr, tpr, thresholds = roc_curve(y_test, model.predict_proba(X_test_piped)[:,1])
# Calculate Area under the curve to display on the plot
    auc = roc_auc_score( y_test, model.predict(X_test_piped) )
# Now, plot the computed values
    plt.plot(fpr, tpr, label ='%s ROC (area = %0.2f)' % (m['label'], auc))
    
# Custom settings for the plot 
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC comparison')
plt.legend(loc="lower right")
plt.show()

# Prediction

In [ ]:
model = XGBClassifier()
model.fit(X_test_piped, y_test)
y_test_pred = model.predict(X_test_piped)

# Evaluation 

**Regression**
* R-Squared
* RMSE : Root Mean Squared Error [0- infinity]
* MAE : Mean Absolute Error [0- infinity]

**Classification**
* Accuracy
*Precision
*Recall
*Specificity
*F1 Score


* R-squared : how well the predictions approximate the  ground truth [0?100%]
*RMSE and MAE are more interpretable than R-squared and Using the RMSE is more appropriate if the data have a lot of outliers.
*Accuracy : This is a metric that is best used for a balanced data set
*Precision : This metric that?s be used to avoid false positive predict (observation is negative, but is predicted positive)
*Recall / Sensitivity : This metric that?s be used to avoid false negative (Observation is positive, but is predicted negative) and maximize True positive rate.
*Specificity : This metric is used to maximize True negative rate and minimize True positive rate.
*F1 Score : This metric that?s be used to minimize False positive and False negative (????? ??? ??? ??? ?????? ???? ???? ?? ???? ?? ???? ?????. ?? ???, ???, ??? ??? ??? ??? ????? ??? ??? ????? ???? ???. ?? ???? ???? ?? ??? F1??? ?????? ??.)




# Evaluatin metrics for regressor models

f1_score(y_test, y_pred, average='micro')

score = mean_absolute_error(y_test, y_pred)
print('MAE:', score)

mse = mean_squared_error(y_test, y_pred)
print('MSE:', mse)

rmse = np.sqrt(mse)
print('RMSE:', rmse)

# Evaluation metrics for classifier models

In [ ]:
pd.DataFrame(confusion_matrix(y_test, y_pred), 
             columns = ['predicted Negative', 'Predicted Positive'],
             index = ['Actual Negative', 'Actual Positive'])

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
X_train.columns

In [ ]:
pd.DataFrame(X_test_piped)

In [ ]:
X_train.dtypes

In [ ]:
importances = model.feature_importances_
features = X_train.columns

feat_importances = pd.Series(importances, index=features)

feat_importances.nlargest(5).sort_values(ascending=True).plot(kind='barh', figsize=(4, 3))

In [ ]:
from sklearn.inspection import permutation_importance

perm_importance = permutation_importance(model, X_test_piped, y_test)

sorted_idx = perm_importance.importances_mean.argsort()
plt.barh(X_train.columns[sorted_idx], perm_importance.importances_mean[sorted_idx])
plt.xlabel("Permutation Importance")

In [ ]:
!pip install shap

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_piped)

In [ ]:
shap.summary_plot(shap_values, 
                  features=X_test_piped, 
                  feature_names=X_train.columns)